# The main purpose of this code is to compute elemental abundances of solar spectra for different instruments

In [ ]:
import os
import sys
import numpy as np
import logging
import multiprocessing
from multiprocessing import Pool
import matplotlib.pyplot as plt
import pandas as pd
from astropy.coordinates import Angle
import astropy.units as u
import math
#from userlist import *
import shutil

################################################################################
#--- iSpec directory -------------------------------------------------------------
#ispec_dir = os.path.dirname(os.path.realpath(__file__)) + "/"
ispec_dir = "/Users/jiayue/iSpec/"
print(ispec_dir)

#ispec_dir = '/home/virtual/iSpec/'
sys.path.insert(0, os.path.abspath(ispec_dir))
import ispec

#--- Change LOG level ----------------------------------------------------------
#LOG_LEVEL = "warning"
LOG_LEVEL = "info"
logger = logging.getLogger() # root logger, common for all
logger.setLevel(logging.getLevelName(LOG_LEVEL.upper()))
################################################################################


## Part 1: Define the solar reference spectrum table

In [ ]:
# Each row corresponds to “one solar reference spectrum for a given instrument”
solar_configs = [
    {
        "instrument": "HARPS",
        "solar_label": "Sun_Dall2006_HARPS",   # used as the target name & part of the output filename
        "spectrum_path": os.path.join(
            ispec_dir,
            "input/spectra/Sun/Sun_DallEtal2006_HARPS.fits"  # the actual location where you place it
        ),
        "unit_is_Angstrom": False,
        "from_resolution": 115000,  # original HARPS resolving power R
        # Initial atmospheric-parameter guesses
        "initial_teff": 5777,
        "initial_logg": 4.44,
        "initial_MH": 0.00,
    },

    # {
    #     "instrument": "MIKE",
    #     "solar_label": "Sun_MIKE_2020XX",
    #     ...
    # },
]


## Part2: data_processing

In [ ]:

from DataPreProcess import (read_write_spectrum,
    parse_obs_time, parse_ra_to_hms, parse_dec_to_dms,
    determine_tellurics_shift_with_mask,
    calculate_barycentric_velocity,
    determine_radial_velocity_with_mask,
    apply_velocity_correction, 
    normalize_whole_spectrum_with_template, 
    plot_spectrum_with_normalization, 
    normalize_spectrum_using_continuum_regions
)

def run_step1_data_processing_for_sun(cfg, norm_method="None"):
    """
    Input: one dict entry from solar_configs
    Output: the normalized spectrum path spectrum_norm_path
    """
    solar_label = cfg["solar_label"]
    input_spectra_path = cfg["spectrum_path"]
    unit_is_Angstrom = cfg["unit_is_Angstrom"]
    from_resolution = cfg["from_resolution"]

    # Set the output path to mirror the stellar case, but replace target with solar_label
    output_folder_Sun = ispec_dir+"output/Sun/"+solar_label+"/"
    os.makedirs(output_folder_Sun, exist_ok=True)

    # ====== 1_data_processing ======
    # ====== read the spectrum ======
    spectra_target_path = input_spectra_path
    spectra_target = read_write_spectrum(spectra_target_path, solar_label, output_folder_Sun, unit_is_Angstrom)
    if np.median(spectra_target["flux"]) < 1e-5:
        spectra_target["flux"] = spectra_target["flux"] * (1/np.max(spectra_target["flux"]))
    # plot original spectrum
    ispec.plot_spectra([spectra_target])

    # ====== determine radial velocity ======
    bv_target, bv_err_target, spectra_after_bv = determine_tellurics_shift_with_mask(spectra_target)
    rv_target, rv_err_target = determine_radial_velocity_with_mask(spectra_after_bv)
    # do the barycentric velocity correction
    spectra_after_bv_rv = ispec.correct_velocity(spectra_after_bv, rv_target)
    # plot spectra after correction
    ispec.plot_spectra([spectra_target, spectra_after_bv, spectra_after_bv_rv])
    # save the final corrected spectrum
    ispec.write_spectrum(spectra_after_bv_rv, output_folder_Sun+solar_label+"_corrected.fits")

    # ====== Degrade the resolution =====
    # No need to degrade the resolution as its the Sun's template

    # ====== Normalization ======
    # 1. Use template to normalize the whole spectrum
    if norm_method == "template":
        star_spectrum_target = ispec.read_spectrum(output_folder_Sun+solar_label+"_corrected.fits")
        spectrum_norm, starcontnmodel = normalize_whole_spectrum_with_template(star_spectrum_target, from_resolution)
        ispec.write_spectrum(spectrum_norm, output_folder_Sun+f"{solar_label}_normalized_template.fits")
        plot_spectrum_with_normalization(star_spectrum_target, spectrum_norm, starcontnmodel)
        return spectrum_norm
    # 2. Use splines in segments to normalize the spectrum
    elif norm_method == "continuum_regions":
        spectrum_norm, starcontnmodel = normalize_spectrum_using_continuum_regions(star_spectrum_target, from_resolution)
        ispec.write_spectrum(spectrum_norm, output_folder_Sun+f"{solar_label}_normalized_continuum_regions.fits")
        plot_spectrum_with_normalization(star_spectrum_target, spectrum_norm, starcontnmodel)
        return spectrum_norm
    else:
        print("No normalization method selected.")
        return None

In [ ]:
run_step1_data_processing_for_sun(solar_configs[0])

- for Sun_UVES_Melendez: `Sun_UVES_Melendez_normed_segmts.fits`
    - input_spectra_path = ispec_dir+"input/spectra/Sun" + solar+label + "/Sun_UVES_Melendez_corrected.fits"
    - splines in segments (`segments-UVES.txt`)
    - wavelength step for median selection: 0.05
    - wavelength step for mac selection: 1.0

## Part3: atmos_params_iteration

In [ ]:
cfg = solar_configs[0]
spectrum_norm_path = ispec_dir + "output/Sun/Sun_UVES_Melendez/Sun_UVES_Melendez_normed_segmts.fits"

In [ ]:
solar_label = cfg["solar_label"]
from_resolution = cfg["from_resolution"]
initial_teff = cfg["initial_teff"]
initial_logg = cfg["initial_logg"]
initial_MH = cfg["initial_MH"]
target = solar_label
output_folder = ispec_dir+"output/Sun/"+target+"/"
instrument = cfg["instrument"]

# For 2_linelist_modify
output_linelist_folder = ispec_dir+"input/linelists/linelist_melendez2014/"
output_linelist_path = output_linelist_folder+"linelist_melendez2014_ispec.tsv"
# For 3_atmos_params_iteration
linelist_target_path = output_folder + "linelist_for_" + target + ".tsv"
#output_linelist_path = "input/linelists/transitions/GESv6_atom_hfs_iso.420_920nm/atomic_lines.tsv"
fit_width = 0.2     # 定义拟合区域宽度 (nm)
# dump_file = "example_results_ew_%s.dump" % (code)
output_atmos_params_dumpfile_path = os.path.join(
    output_folder,
    f"atmos_params_{target}.dump"
)


In [ ]:
# ====== 1. Check Gaussian fits for iron lines ======
# =============== (1) Get paths & read data =================
# get the spectrum and linelist path
star_spectrum_file = spectrum_norm_path
star_spectrum = ispec.read_spectrum(star_spectrum_file)
# copy the linelist file to a new location
shutil.copyfile(output_linelist_path, linelist_target_path)
print(f"Copied {output_linelist_path} -> {linelist_target_path}")
atomic_linelist_file = linelist_target_path
print(atomic_linelist_file)

# =============== (2) Read linelist and restrict wavelength range =================
# read (start to find lines)
logging.info("Finding line masks...")
atomic_linelist = ispec.read_atomic_linelist(atomic_linelist_file)  # read the full linelist (optional)
print(f"Successfully read {len(atomic_linelist)} lines from the linelist.")
# Select only lines within the actual wavelength range of the spectrum.
# For example, if star_spectrum covers 480–680 nm, keep only lines in this range.
atomic_linelist = ispec.read_atomic_linelist(
    atomic_linelist_file,
    wave_base=np.min(star_spectrum['waveobs']),
    wave_top=np.max(star_spectrum['waveobs'])
)
#atomic_linelist = atomic_linelist[atomic_linelist['theoretical_depth'] >= 0.01] # Select lines with minimal contribution in the Sun
print(
    f"Successfully read {len(atomic_linelist)} lines from the linelist in range from "
    f"{np.min(star_spectrum['waveobs'])} to {np.max(star_spectrum['waveobs'])}."
)
print("Linelist wavelength range:", np.min(atomic_linelist['wave_nm']), "to", np.max(atomic_linelist['wave_nm']), "nm.")

# =============== Spectral convolution (smoothing) + continuum fitting ===============
smoothed_star_spectrum = star_spectrum
star_continuum_model = ispec.fit_continuum(star_spectrum, fixed_value=1.0, model="Fixed value")

# Telluric
telluric_linelist = None
vel_telluric = 0.0
min_depth = 0.05    # minimum absorption depth for line detection (not strictly required)
max_depth = 1.00

# =============== Line finding + Gaussian fitting ===============
# atomic_linelist: atomic line list used for cross-matching
# max_atomic_wave_diff: allowed wavelength mismatch for line centers, in nm
# discard_voigt=True: use only Gaussian profiles
# min_depth=0.05: consider only lines deeper than 5%
# closest_match=False: match lines using theoretical parameters
star_linemasks = ispec.find_linemasks(
    star_spectrum,
    star_continuum_model,
    atomic_linelist=atomic_linelist,
    max_atomic_wave_diff=0.005,
    telluric_linelist=telluric_linelist,
    vel_telluric=vel_telluric,
    minimum_depth=min_depth,
    maximum_depth=max_depth,
    smoothed_spectrum=smoothed_star_spectrum,
    check_derivatives=False,
    discard_gaussian=False,
    discard_voigt=True,
    closest_match=False
)
# star_linemasks is a structured array containing fitted line parameters (mu, sigma, A, etc.),
# equivalent width (ew), rms, depth, element name, ionization stage, loggf, and cross-matching info

# =============== Filter invalid spectral lines ===============
# Exclude lines that were not successfully cross-matched with the atomic linelist,
# because chemical abundances cannot be computed for them (would crash downstream routines)
# Remove lines not matched to atomic linelist (wave_nm = 0)
rejected_by_atomic_line_not_found = (star_linemasks['wave_nm'] == 0)
star_linemasks = star_linemasks[~rejected_by_atomic_line_not_found]

# Exclude lines with EW equal to zero
# Remove lines with zero equivalent width (failed fit or no absorption feature)
rejected_by_zero_ew = (star_linemasks['ew'] == 0)
star_linemasks = star_linemasks[~rejected_by_zero_ew]

# ========== Keep only lines with EW between 20 and 100 mÅ ==========
ew_range_mask = (star_linemasks['ew'] >= 20) & (star_linemasks['ew'] <= 100)
star_linemasks = star_linemasks[ew_range_mask]

# =============== Optional: select Fe lines only (for Fe abundance analysis) ===============
# Select only iron lines
# If a custom linelist is already used, this step may be unnecessary
iron = star_linemasks['element'] == "Fe 1"
iron = np.logical_or(iron, star_linemasks['element'] == "Fe 2")
iron_star_linemasks = star_linemasks[iron]

# =============== Write linemasks files ===============
linemask_output_folder = output_folder + "/linemasks"
os.makedirs(linemask_output_folder, exist_ok=True)
# Write regions with only mask limits and notes:
# original line regions, containing only base/peak/top information
ispec.write_line_regions(
    star_linemasks,
    linemask_output_folder+"/"+target+"_melendez2014_star_linemasks.txt"
)
# Write iron regions with only mask limits and notes:
# only Fe lines with base/peak/top information
ispec.write_line_regions(
    iron_star_linemasks,
    linemask_output_folder+"/"+target+"_melendez2014_star_fe_linemasks.txt"
)
# Write regions with mask limits, cross-matched atomic data, and fit data:
# includes cross-matching and fit information, for use with model_spectrum_from_ew
ispec.write_line_regions(
    star_linemasks,
    linemask_output_folder+"/"+target+"_melendez2014_star_fitted_linemasks.txt",
    extended=True
)
recover_star_linemasks = ispec.read_line_regions(
    linemask_output_folder+"/"+target+"_melendez2014_star_fitted_linemasks.txt"
)
# Write regions with mask limits and cross-matched atomic data (fit data fields zeroed):
# all fit information is reset, but cross-matching is preserved for batch simulations
zeroed_star_linemasks = ispec.reset_fitted_data_fields(star_linemasks)
ispec.write_line_regions(
    zeroed_star_linemasks,
    linemask_output_folder+"/"+target+"_melendez2014_star_zeroed_fitted_linemasks.txt",
    extended=True
)

# Write only atomic data for the selected regions:
# save the cross-matched atomic linelist
ispec.write_atomic_linelist(
    star_linemasks,
    linemask_output_folder+"/"+target+"_melendez2014_star_atomic_linelist.txt"
)

In [ ]:
# =============== Plotting ===============
# Create output directories
output_dir = output_folder + "/figs_Fe_GaussianFits"
os.makedirs(output_dir, exist_ok=True)
deleted_felines_folder_path = output_dir+"/deleted"
os.makedirs(deleted_felines_folder_path, exist_ok=True)

# Keep only Fe I and Fe II lines
fe_lines = star_linemasks[
    (star_linemasks['element'] == "Fe 1") | 
    (star_linemasks['element'] == "Fe 2")
]

w_range = 0.25  # unit: nm

# Define the Gaussian function
def gaussian(x, mu, sig, A, baseline):
    return baseline + A * np.exp(-(x - mu)**2 / (2 * sig**2))

# Loop over each line and make plots
for idx, line in enumerate(fe_lines):
    mu = line['mu']
    sig = line['sig']
    A = line['A']
    baseline = line['baseline']
    
    if sig == 0 or mu == 0:  # invalid fit
        continue

    # Select the wavelength segment from the original spectrum
    mask = (star_spectrum['waveobs'] >= mu - w_range) & (star_spectrum['waveobs'] <= mu + w_range)
    wave = star_spectrum['waveobs'][mask]
    flux = star_spectrum['flux'][mask]

    # Construct the fitted curve
    fit_x = np.linspace(mu - w_range, mu + w_range, 300)
    fit_y = gaussian(fit_x, mu, sig, A, baseline)

    # Plot
    plt.figure(figsize=(8, 5), dpi=128)
    plt.plot(wave, flux, label='Observed Spectrum', color='blue', lw=0.7)
    plt.plot(fit_x, fit_y, '--', label='Gaussian Fit', color='red')
    plt.axvline(mu, color='orange', linestyle=':', label=f"$\mu$ = {mu:.3f} nm")
    plt.title(f"Gaussian Fit: {line['element']} {line['wave_A']:.2f} Å", fontsize=16)
    plt.xlabel("Wavelength (nm)", fontsize=16)
    plt.ylabel("Normalized Flux", fontsize=16)
    plt.xticks(fontsize=13)
    plt.yticks(fontsize=13)
    plt.ticklabel_format(style='plain', axis='x')   # disable scientific notation on x-axis

    # Build annotation text
    text = (
        f"log(gf) = {line['loggf']:.2f}\n"
        f"EP = {line['lower_state_eV']:.2f} eV\n"
        f"EW = {line['ew']:.1f} mÅ"
    )
    plt.text(
        0.75, 0.05, text, transform=plt.gca().transAxes,
        fontsize=13, bbox=dict(facecolor='white', alpha=0.8)
    )
    plt.legend(loc="lower left", fontsize=13)
    plt.tight_layout()
    
    # Save figure
    out_path = os.path.join(output_dir, f"FeFit_{idx+1}_{line['element']}_{line['wave_A']:.2f}.png")
    plt.savefig(out_path)
    plt.close()

print(f"Plotting completed: {len(fe_lines)} figures generated. Results saved in {output_dir}/")

In [ ]:
# ======== Create tables for manually recording deleted lines =========
shutil.copyfile(output_linelist_path, output_folder + "linemasks/linelist_for_" + target + "_copied.tsv")
deleted_file_path = output_folder + "linemasks/Lines_deleted.tsv"
modified_file_path = output_folder + "linemasks/Lines_EW_modified.tsv"

# Create the linemasks directory
os.makedirs(os.path.dirname(deleted_file_path), exist_ok=True)
os.makedirs(os.path.dirname(modified_file_path), exist_ok=True)

# Full header for deleted.tsv
cols_deleted = [
    "element","wave_A","wave_nm","loggf","lower_state_eV","lower_state_cm1","lower_j",
    "upper_state_eV","upper_state_cm1","upper_j","upper_g","lande_lower","lande_upper",
    "spectrum_transition_type","turbospectrum_rad","rad","stark","waals",
    "waals_single_gamma_format","turbospectrum_fdamp","spectrum_fudge_factor",
    "theoretical_depth","theoretical_ew","lower_orbital_type","upper_orbital_type",
    "molecule","spectrum_synthe_isotope","ion","spectrum_moog_species",
    "turbospectrum_species","width_species","reference_code","spectrum_support",
    "turbospectrum_support","moog_support","width_support","synthe_support","sme_support"
]

# Header for modified.tsv
cols_modified = ["element", "wave_A", "wave_nm", "loggf", "ew_mA_new"]

def ensure_tsv_with_header(path, columns):
    if not os.path.exists(path):
        pd.DataFrame(columns=columns).to_csv(path, sep="\t", index=False)
        print(f"Created empty file with header: {path}")
    else:
        print(f"File already exists: {path}")

ensure_tsv_with_header(deleted_file_path,  cols_deleted)
ensure_tsv_with_header(modified_file_path, cols_modified)
print("\nIf you have manually deleted or modified any lines in the linemasks files,")
print(f"please document the changes in:\n  {deleted_file_path}\n  {modified_file_path}")
print("If no manual changes were made, you can ignore this message.")

In [ ]:
# ====== 2. Filter the linelist based on the manually deleted-lines table ======
# Original linelist file
atomic_linelist_file = linelist_target_path
# Output file
filtered_linelist_file = linelist_target_path

# 1. Read the original linelist
atomic_linelist = ispec.read_atomic_linelist(atomic_linelist_file)
print("Number of lines in the original linelist:", len(atomic_linelist))

# 2. Define wavelengths (nm) to be removed
# Read the lines to be deleted
df_deleted = pd.read_csv(output_folder+"linemasks/Lines_deleted.tsv", sep="\t")
# Normalize columns and filter invalid values
df_deleted["element"] = df_deleted["element"].astype(str)
df_deleted["wave_nm"] = pd.to_numeric(df_deleted["wave_nm"], errors="coerce")
df_deleted = df_deleted.dropna(subset=["wave_nm"])
# Separate Fe and non-Fe lines
is_fe = df_deleted["element"].str.startswith("Fe")
fe_lines    = df_deleted.loc[is_fe,  "wave_nm"].round(4).tolist()
other_lines = df_deleted.loc[~is_fe, "wave_nm"].round(4).tolist()
# Build the final list (Fe first, then others; deduplicate while preserving order)
from collections import OrderedDict
lines_to_remove = list(OrderedDict.fromkeys(fe_lines + other_lines))

# 3. Use a boolean mask to keep rows not listed for deletion
mask = ~np.isin(np.round(atomic_linelist['wave_nm'], 4), np.round(lines_to_remove, 4))
filtered_linelist = atomic_linelist[mask]
print("Number of lines after filtering:", len(filtered_linelist))

# 4. Write back the new file
ispec.write_atomic_linelist(filtered_linelist, filtered_linelist_file)
print(f"New linelist saved to: {filtered_linelist_file}")

# 5. Check whether it can be read correctly
test_linelist = ispec.read_atomic_linelist(filtered_linelist_file)
print("New linelist read successfully, number of lines:", len(test_linelist))

atomic_linelist_file = linelist_target_path

# ===== Rewrite linemasks files =====
# =============== 2. Read the linelist and restrict the wavelength range =================
# read (start to find lines)
logging.info("Finding line masks...")
atomic_linelist = ispec.read_atomic_linelist(atomic_linelist_file)  # Read the full linelist (optional)
print(f"Successfully read {len(atomic_linelist)} lines from the linelist.")
# Filter lines according to the actual wavelength range of the spectrum.
# For example, if star_spectrum spans from 480 nm to 680 nm, keep only this range.
atomic_linelist = ispec.read_atomic_linelist(
    atomic_linelist_file,
    wave_base=np.min(star_spectrum['waveobs']),
    wave_top=np.max(star_spectrum['waveobs'])
)
# atomic_linelist = atomic_linelist[atomic_linelist['theoretical_depth'] >= 0.01]  # Select lines with minimal contribution in the Sun
print(
    f"Successfully read {len(atomic_linelist)} lines from the linelist in range "
    f"from {np.min(star_spectrum['waveobs'])} to {np.max(star_spectrum['waveobs'])}."
)
print(
    "Linelist wavelength range:",
    np.min(atomic_linelist['wave_nm']),
    "to",
    np.max(atomic_linelist['wave_nm']),
    "nm."
)

# =============== Line finding + Gaussian fitting ===============
# atomic_linelist: atomic linelist used for cross-matching
# max_atomic_wave_diff: allowed tolerance for line-center matching (nm)
# discard_voigt=True: use Gaussian fitting only
# min_depth=0.05: consider only lines deeper than 5%
# closest_match=False: match lines using theoretical parameters
star_linemasks = ispec.find_linemasks(
    star_spectrum,
    star_continuum_model,
    atomic_linelist=atomic_linelist,
    max_atomic_wave_diff=0.005,
    telluric_linelist=telluric_linelist,
    vel_telluric=vel_telluric,
    minimum_depth=min_depth,
    maximum_depth=max_depth,
    smoothed_spectrum=smoothed_star_spectrum,
    check_derivatives=False,
    discard_gaussian=False,
    discard_voigt=True,
    closest_match=False
)
# star_linemasks is a structured array containing fitted line parameters (mu, sigma, A, etc.),
# equivalent width (ew), rms, depth, element name, ionization stage, loggf, and cross-matching information.

# =============== Filter invalid lines ===============
# Exclude lines that have not been successfully cross-matched with the atomic data
# because chemical abundances cannot be computed for them (the routines would crash).
# Remove lines not matched to the atomic linelist (wave_nm = 0)
rejected_by_atomic_line_not_found = (star_linemasks['wave_nm'] == 0)
star_linemasks = star_linemasks[~rejected_by_atomic_line_not_found]

# Exclude lines with EW equal to zero
# Remove lines with zero equivalent width (fit failed or no absorption feature)
rejected_by_zero_ew = (star_linemasks['ew'] == 0)       # EW range used for line selection
star_linemasks = star_linemasks[~rejected_by_zero_ew]

# ========== Keep only lines with EW between 20 and 100 mÅ ==========
ew_range_mask = (star_linemasks['ew'] >= 20) & (star_linemasks['ew'] <= 100)
star_linemasks = star_linemasks[ew_range_mask]

# =============== Optional: select Fe lines only (for Fe abundance analysis) ===============
# Select only iron lines. If you manually selected the linelist,
# this step may not be necessary and all lines can be used directly.
iron = star_linemasks['element'] == "Fe 1"
iron = np.logical_or(iron, star_linemasks['element'] == "Fe 2")
iron_star_linemasks = star_linemasks[iron]

# =============== Write linemasks files ===============
linemask_output_folder = output_folder + "/linemasks"
os.makedirs(linemask_output_folder, exist_ok=True)
# Write regions with only mask limits and notes:
# Original line segments, containing only base/peak/top information
ispec.write_line_regions(
    star_linemasks,
    linemask_output_folder+"/"+target+"_melendez2014_star_linemasks.txt"
)
# Write iron regions with only mask limits and notes:
# Only Fe lines with base/peak/top information
ispec.write_line_regions(
    iron_star_linemasks,
    linemask_output_folder+"/"+target+"_melendez2014_star_fe_linemasks.txt"
)
# Write regions with mask limits, cross-matched atomic data, and fit data
# Includes cross-matching and fit information, used by model_spectrum_from_ew
ispec.write_line_regions(
    star_linemasks,
    linemask_output_folder+"/"+target+"_melendez2014_star_fitted_linemasks.txt",
    extended=True
)
recover_star_linemasks = ispec.read_line_regions(
    linemask_output_folder+"/"+target+"_melendez2014_star_fitted_linemasks.txt"
)
# Write regions with mask limits and cross-matched atomic data (fit fields zeroed)
# All fit information is reset, but cross-matching is preserved for batch simulations
zeroed_star_linemasks = ispec.reset_fitted_data_fields(star_linemasks)
ispec.write_line_regions(
    zeroed_star_linemasks,
    linemask_output_folder+"/"+target+"_melendez2014_star_zeroed_fitted_linemasks.txt",
    extended=True
)

# Write only atomic data for the selected regions:
# Save the cross-matched atomic linelist
ispec.write_atomic_linelist(
    star_linemasks,
    linemask_output_folder+"/"+target+"_melendez2014_star_atomic_linelist.txt"
)

## 3.2 Calculate the abundances of Fe I and Fe II

In [ ]:
# ========== Read the spectrum ==========
# --- spectrum that has already been generated previously ---
# star_spectrum: normalized spectrum
# star_continuum_model: continuum model obtained earlier using fit_continuum

# ========== Read the saved fitted linemasks ==========
linemasks = ispec.read_line_regions(output_folder + f"/linemasks/{target}_melendez2014_star_fitted_linemasks.txt")

# Filter invalid lines:
linemasks = linemasks[linemasks['wave_nm'] > 0]   # successfully matched lines
linemasks = linemasks[linemasks['ew'] > 0]        # lines with valid EW

# --- Determining abundances by EW of the previously fitted lines ---------------
code="moog"
# Parameters
teff = initial_teff
logg = initial_logg
MH = initial_MH
alpha = 0.00
microturbulence_vel = 1.0

# ========== Read model atmospheres and solar abundances ==========
# Selected model atmosphere grid and solar abundances
# Atmosphere models (different authors use different grids; differences are mainly in speed)
#model = ispec_dir + "/input/atmospheres/MARCS/"    # very large grid
model = ispec_dir + "/input/atmospheres/MARCS.GES/" # smaller grid with preliminary interpolation
#model = ispec_dir + "/input/atmospheres/MARCS.APOGEE/"
#model = ispec_dir + "/input/atmospheres/ATLAS9.APOGEE/"
#model = ispec_dir + "/input/atmospheres/ATLAS9.Castelli/"
#model = ispec_dir + "/input/atmospheres/ATLAS9.Kurucz/"
#model = ispec_dir + "/input/atmospheres/ATLAS9.Kirby/"

# Solar abundance set (different authors provide different solar compositions)
if "ATLAS" in model:
    solar_abundances_file = ispec_dir + "/input/abundances/Grevesse.1998/stdatom.dat"
else:
    # MARCS
    solar_abundances_file = ispec_dir + "/input/abundances/Grevesse.2007/stdatom.dat"
#solar_abundances_file = ispec_dir + "/input/abundances/Asplund.2005/stdatom.dat"
#solar_abundances_file = ispec_dir + "/input/abundances/Asplund.2009/stdatom.dat"
#solar_abundances_file = ispec_dir + "/input/abundances/Anders.1989/stdatom.dat"

# ========== Load models ==========
# Load model atmosphere grid
modeled_layers_pack = ispec.load_modeled_layers_pack(model)
# Load solar abundances
solar_abundances = ispec.read_solar_abundances(solar_abundances_file)

# Validate parameters
# Check whether the parameters are within the model grid
if not ispec.valid_atmosphere_target(modeled_layers_pack, {'teff':teff, 'logg':logg, 'MH':MH, 'alpha':alpha}):
    msg = "The specified effective temperature, gravity (log g), and metallicity [M/H] \
            fall outside the atmospheric model grid."
    print(msg)

# Prepare atmosphere model
# Interpolate the model atmosphere
atmosphere_layers = ispec.interpolate_atmosphere_layers(
    modeled_layers_pack,
    {'teff':teff, 'logg':logg, 'MH':MH, 'alpha':alpha},
    code=code
)

# ========== Run abundance determination ==========
spec_abund, normal_abund, x_over_h, x_over_fe = ispec.determine_abundances(
    atmosphere_layers,
    teff, logg, MH, alpha,
    linemasks,
    solar_abundances,
    microturbulence_vel=microturbulence_vel,
    verbose=1,
    code=code
)

# ========== Summarize Fe abundances ==========
bad = np.isnan(x_over_h)
fe1 = linemasks['element'] == "Fe 1"
fe2 = linemasks['element'] == "Fe 2"
fe1_abund = x_over_h[np.logical_and(fe1, ~bad)]
fe2_abund = x_over_h[np.logical_and(fe2, ~bad)]
print("\n===== [Fe/H] reference =====")
print(f"[Fe/H]: {MH} +- 0.00 (input value)")

print("\n===== Fe I =====")
print("Number of Fe I lines:", len(fe1_abund))
print(
    "[Fe 1/H] median: %.4f" % np.median(fe1_abund),
    ", mean: %.4f" % np.mean(fe1_abund),
    ", std: %.4f" % np.std(fe1_abund)
)
print(fe1_abund)

print("\n===== Fe II =====")
print("Number of Fe II lines:", len(fe2_abund))
print(
    "[Fe 2/H] median: %.4f" % np.median(fe2_abund),
    ", mean: %.4f" % np.mean(fe2_abund),
    ", std: %.4f" % np.std(fe2_abund)
)
print(fe2_abund)

spec_abund_fe1 = spec_abund[np.logical_or(fe1, fe2)]
print(len(spec_abund_fe1), "Fe lines used for abundance analysis.")

In [ ]:
# Find the indices where [Fe II/H] equals a given target value
target_abund = 0.003
fe2_mask = (linemasks['element'] == "Fe 2") & (~np.isnan(x_over_h))
fe2_indices = np.where(fe2_mask)[0]
target_idx = fe2_indices[np.isclose(x_over_h[fe2_mask], target_abund, atol=1e-4)]

# Output the corresponding spectral line information
if len(target_idx) > 0:
    print(linemasks[target_idx][['wave_nm', 'element', 'ew', 'loggf']])
else:
    print("No matching spectral line found")

## 3.3 Atmospheric parameters fitting

In [ ]:
atmos_iter_method = "lbl"
# "lbl": line-by-line method
# "glb": global fitting method

In [ ]:
# --- Read the normalized spectrum -------------------------------------------------------------
spectrum_path = spectrum_norm_path
normalized_star_spectrum = ispec.read_spectrum(spectrum_path)

if atmos_iter_method == "glb":
    print("\n--- Model spectra from EW + Global Fitting --------------------------------------------------")
    # --- Model spectra from EW --------------------------------------------------
    # Parameters
    initial_teff = initial_teff  # row_target["Teff"]
    initial_logg = initial_logg  # row_target["logg"]
    initial_MH = initial_MH      # row_target["[Fe/H]"]
    initial_alpha = 0.00
    initial_vmic = ispec.estimate_vmic(initial_teff, initial_logg, initial_MH)
    max_iterations = 15

    # Selected model atmosphere, linelist, and solar abundances

    # model = ispec_dir + "/input/atmospheres/MARCS.GES/"
    # solar_abundances_file = ispec_dir + "/input/abundances/Grevesse.2007/stdatom.dat"
    # Load model atmospheres
    modeled_layers_pack = ispec.load_modeled_layers_pack(model)

    # Load SPECTRUM abundances
    solar_abundances = ispec.read_solar_abundances(solar_abundances_file)

    # Validate parameters
    if not ispec.valid_atmosphere_target(
        modeled_layers_pack,
        {'teff': initial_teff, 'logg': initial_logg, 'MH': initial_MH, 'alpha': initial_alpha}
    ):
        msg = "The specified effective temperature, gravity (log g), and metallicity [M/H] " \
              "fall outside the atmospheric model grid."
        print(msg)

    results = ispec.model_spectrum_from_ew(
        linemasks, modeled_layers_pack,
        solar_abundances, initial_teff, initial_logg, initial_MH, initial_alpha, initial_vmic,
        free_params=["teff", "logg", "vmic"],
        adjust_model_metalicity=True,
        max_iterations=max_iterations,
        enhance_abundances=True,
        outliers_detection="robust",
        outliers_weight_limit=0.90,
        # outliers_detection='sigma_clipping',
        # sigma_level=3,
        tmp_dir=None,
        code=code
    )
    params, errors, status, x_over_h, selected_x_over_h, fitted_lines_params, used_linemasks = results  # output

    teff_1, logg_1, mh_1, vmic_1, alpha_1 = (
        params['teff'], params['logg'], params['MH'], params['vmic'], params['alpha']
    )
    print("Teff = ", teff_1, "K", ", error:", errors['teff'])
    print("logg = ", logg_1, ", error:", errors['logg'])
    print("[M/H] = ", mh_1, ", error:", errors['MH'])
    print("vmic = ", vmic_1, "km/s", ", error:", errors['vmic'])

    print("Reference parameters:", f"Teff = {initial_teff}, logg = {initial_logg}, MH = {initial_MH}")

    bad2 = np.isnan(x_over_h)
    spec_abund_fe = spec_abund_fe1[~bad2]
    print(len(spec_abund_fe), "Fe lines used for abundance analysis.")

elif atmos_iter_method == "lbl":
    from AbundProcessFunctions import export_q2_inputs_single_wide
    print("\n--- Model spectra from EW + Line-by-Line Fitting --------------------------------------------------")
    # --- Model spectra from EW --------------------------------------------------
    # ================== Derive solar atmospheric parameters with q2 (Fe I / Fe II line-by-line) ==================
    import q2

    # 1) Keep only Fe lines (in iSpec usually labeled as "Fe 1" / "Fe 2")
    elem_col = np.array([str(e).strip() for e in linemasks["element"]])
    linemasks_fe = linemasks[(elem_col == "Fe 1") | (elem_col == "Fe 2")]

    # 2) Prepare q2 working directory
    q2_dir = os.path.join(output_folder, "q2_work")
    os.makedirs(q2_dir, exist_ok=True)

    # 3) Generate stars.csv / lines.csv required by q2
    initial_vmic = ispec.estimate_vmic(initial_teff, initial_logg, initial_MH)
    stars_csv, lines_csv = export_q2_inputs_single_wide(
        linemasks_fe,
        star_id=solar_label,
        teff=initial_teff,
        logg=initial_logg,
        feh=initial_MH,
        vt=float(initial_vmic),
        out_dir=q2_dir,
        fe_only=True,
    )

    # 4) Run q2
    os.chdir(q2_dir)
    data = q2.Data(stars_csv, lines_csv)
    sp = q2.specpars.SolvePars(grid="marcs")
    solution_csv = os.path.join(q2_dir, "solution.csv")
    q2.specpars.solve_all(data, sp, solution_csv)

    sol = pd.read_csv(solution_csv)
    row = sol.loc[sol["id"] == str(solar_label)].iloc[0]
    teff_q2, logg_q2, feh_q2, vt_q2 = map(float, [row["teff"], row["logg"], row["feh"], row["vt"]])
    print("[q2 Sun solution]", teff_q2, logg_q2, feh_q2, vt_q2)

else:
    print("Unknown atmosphere iteration method:", atmos_iter_method)

In [ ]:
if atmos_iter_method == "glb":
    #--- Model spectra from EW --------------------------------------------------
    # Parameters
    initial_teff = teff_1
    initial_logg = logg_1
    initial_MH = mh_1
    initial_alpha = alpha_1
    initial_vmic = vmic_1
    max_iterations = 10

    # Selected model amtosphere, linelist and solar abundances

    #model = ispec_dir + "/input/atmospheres/MARCS.GES/"
    #solar_abundances_file = ispec_dir + "/input/abundances/Grevesse.2007/stdatom.dat"
    # Load model atmospheres
    modeled_layers_pack = ispec.load_modeled_layers_pack(model)

    # Load SPECTRUM abundances
    solar_abundances = ispec.read_solar_abundances(solar_abundances_file)

    # Validate parameters
    if not ispec.valid_atmosphere_target(modeled_layers_pack, {'teff':initial_teff, 'logg':initial_logg, 'MH':initial_MH, 'alpha':initial_alpha}):
        msg = "The specified effective temperature, gravity (log g) and metallicity [M/H] \
                fall out of theatmospheric models."
        print(msg)

    results = ispec.model_spectrum_from_ew(used_linemasks, modeled_layers_pack, \
                        solar_abundances, initial_teff, initial_logg, initial_MH, initial_alpha, initial_vmic, \
                        free_params=["teff", "logg", "vmic"], \
                        adjust_model_metalicity=True, \
                        max_iterations=max_iterations, \
                        enhance_abundances=True, \
                        outliers_detection = "robust", \
                        outliers_weight_limit = 0.90, \
                        #outliers_detection = 'sigma_clipping', \
                        #sigma_level = 3, \
                        tmp_dir = None, \
                        code=code)
    params, errors, status, x_over_h, selected_x_over_h, fitted_lines_params, used_linemasks = results  # output

    print("Teff = ", params['teff'], "K", ", error:", errors['teff'])
    print("logg = ", params['logg'], ", error:", errors['logg'])
    print("[M/H] = ", params['MH'], ", error:", errors['MH'])
    print("vmic = ", params['vmic'], "km/s", ", error:", errors['vmic'])

else:
    print("No further iteration performed.")

In [ ]:
def plot_abundance_diagnostics(x_over_h, used_linemasks, selected_x_over_h, element, x_over_h_vs_EP=True, x_over_h_vs_REW=True, savefig=False):
    """
    Note:
    1. This function assumes that `x_over_h`, `used_linemasks`, and `selected_x_over_h`
       all come from the output of `ispec.model_spectrum_from_ew`.
    2. `x_over_h` may contain NaN values (corresponding to lines that cannot be measured),
       so invalid values are removed first using `valid_mask = ~np.isnan(x_over_h)`.
       `df["abund"]` corresponds one-to-one with `used_linemasks`.
    3. `selected_x_over_h` is a boolean mask (with the same length as `x_over_h`),
       indicating whether each line is selected by the EW fitting.
       This function uses `selected_x_over_h[i][valid_mask]`
       to distinguish between selected and outlier lines.
    4. The output plots contain two components:
       - Blue points: lines used in the fit (selected)
       - Red points: rejected lines (outliers)

    5. This function currently supports only "Fe 1" and "Fe 2",
       because `selected_x_over_h` provides masks only for these two species.
    """
    # --- Build DataFrame: include all valid lines ---
    valid_mask = ~np.isnan(x_over_h)  # remove NaN values
    df = pd.DataFrame(used_linemasks)  # same length as valid lines
    df["abund"] = x_over_h[valid_mask]  # assign abundances to all lines

    # Construct the "selected" flag column
    if element == "Fe 1":
        df["selected"] = selected_x_over_h[0][valid_mask]
    elif element == "Fe 2":
        df["selected"] = selected_x_over_h[1][valid_mask]
    else:
        raise ValueError("element has to be 'Fe 1' or 'Fe 2'")

    # --- Split data ---
    ele_df = df[df["element"] == element]
    sel = ele_df[ele_df["selected"] == True]
    out = ele_df[ele_df["selected"] == False]
    print(f"{element} total lines: {len(ele_df)}, selected lines: {len(sel)}, outlier lines: {len(out)}")

    if sel.empty:
        print(f"No selected data found for {element}")
        return None

    # -------- Plot 1: [X/H] vs EP --------
    if x_over_h_vs_EP:
        plt.figure(figsize=(8,6), dpi=192)
        plt.scatter(out["lower_state_eV"], out["abund"], color="red", label="outliers")
        plt.scatter(sel["lower_state_eV"], sel["abund"], color="blue", label="selected lines")
        plt.xlabel('Excitation Potential (eV)', fontsize=14)
        plt.ylabel(f'[{element}/H]', fontsize=14)
        plt.title(f'[{element}/H] vs. EP', fontsize=16)
        plt.legend(fontsize=12)
        plt.grid(linewidth=0.3, linestyle='--', alpha=0.7)
        if savefig:
            output_dir = output_folder + "/figs_atmos_params_Fe_EW"
            os.makedirs(output_dir, exist_ok=True)
            plt.savefig(output_dir + f"/{element}_abundance_vs_EP.png")

    # -------- Plot 2: [X/H] vs REW --------
    if x_over_h_vs_REW:
        plt.figure(figsize=(8,6), dpi=192)
        plt.scatter(sel["ewr"], sel["abund"], color="blue", label=f"{element} selected")
        plt.scatter(out["ewr"], out["abund"], color="red", label=f"{element} outliers")
        plt.xlabel('log(EW / λ)', fontsize=14)
        plt.ylabel(f'[{element}/H]', fontsize=14)
        plt.title(f'[{element}/H] vs. REW', fontsize=16)
        plt.legend(fontsize=12)
        plt.grid(linewidth=0.3, linestyle='--', alpha=0.7)
        if savefig:
            output_dir = output_folder + "/figs_atmos_params_Fe_EW"
            os.makedirs(output_dir, exist_ok=True)
            plt.savefig(output_dir + f"/{element}_abundance_vs_REW.png")

def plot_abundance_vs_EP_with_fit(x_over_h, used_linemasks, selected_x_over_h, element="Fe 1", savefig=False):
    """
    Plot [X/H] vs EP based on the output of ispec.model_spectrum_from_ew and perform a linear fit.

    Parameters:
        x_over_h: abundance array for all lines
        used_linemasks: line masks (list of dicts), containing 'element', 'lower_state_eV', 'ewr', etc.
        selected_x_over_h: list containing [Fe I selected mask, Fe II selected mask]
        element: "Fe 1" or "Fe 2"
    """

    # --- Build DataFrame: include all valid lines ---
    valid_mask = ~np.isnan(x_over_h)  # remove NaN values
    df = pd.DataFrame(used_linemasks)  # same length as valid lines
    df["abund"] = x_over_h[valid_mask]  # assign abundances to all lines

    # Construct the "selected" flag column
    if element == "Fe 1":
        df["selected"] = selected_x_over_h[0][valid_mask]
    elif element == "Fe 2":
        df["selected"] = selected_x_over_h[1][valid_mask]
    else:
        raise ValueError("element has to be 'Fe 1' or 'Fe 2'")

    # --- Split data ---
    ele_df = df[df["element"] == element]
    sel = ele_df[ele_df["selected"] == True]
    out = ele_df[ele_df["selected"] == False]
    print(f"{element} total lines: {len(ele_df)}, selected lines: {len(sel)}, outlier lines: {len(out)}")

    if sel.empty:
        print(f"No selected data found for {element}")
        return None

    # --- Linear fit: [X/H] ~ EP ---
    X = sel["lower_state_eV"].values
    y = sel["abund"].values
    slope, intercept = np.polyfit(X, y, 1)

    # Compute R²
    y_pred = slope * X + intercept
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    # --- Plot ---
    plt.figure(figsize=(8,6), dpi=192)
    plt.scatter(out["lower_state_eV"], out["abund"], color="red", label="outliers")
    plt.scatter(sel["lower_state_eV"], sel["abund"], color="blue", label="selected lines")

    # Regression line
    x_fit = np.linspace(min(ele_df["lower_state_eV"]), max(ele_df["lower_state_eV"]), 100)
    y_fit = slope * x_fit + intercept
    plt.plot(x_fit, y_fit, "k--", lw=2, label=f"Fit: y={slope:.3f}x+{intercept:.3f}\nR²={r2:.3f}")

    plt.xlabel("Excitation Potential (eV)", fontsize=14)
    plt.ylabel(f"[{element}/H] (dex)", fontsize=14)
    plt.title(f"[{element}/H] vs EP with Linear Fit", fontsize=16)
    plt.legend(fontsize=12)
    plt.grid(linewidth=0.3, linestyle='--', alpha=0.7)

    if savefig == True:
        output_dir = output_folder + "/figs_atmos_params_Fe_EW"
        os.makedirs(output_dir, exist_ok=True)
        plt.savefig(output_dir + f"/{element}_abundance_vs_EP_fit.png")

    print(f"{element} fit results: slope = {slope:.4f}, intercept = {intercept:.4f}, R² = {r2:.3f}")
    # ---- Temperature diagnostic ----
    if slope > 0:
        print("Teff may be underestimated (temperature should be increased)")
    elif slope < 0:
        print("Teff may be overestimated (temperature should be decreased)")
    else:
        print("Teff is reasonable, with no significant bias")

    return slope, intercept, r2

In [ ]:
if atmos_iter_method == "glb":
    ##--- Save results -------------------------------------------------------------
    logging.info("Saving results...")
    # dump_file = "example_results_ew_%s.dump" % (code)
    dump_file = output_atmos_params_dumpfile_path
    ispec.mkdir_p(os.path.dirname(dump_file))
    ispec.save_results(
        dump_file,
        (params, errors, status, x_over_h, selected_x_over_h,
         fitted_lines_params, used_linemasks, spec_abund_fe)
    )
    # If needed, results can be restored in another script using:
    # params, errors, status, x_over_h, selected_x_over_h, fitted_lines_param, used_linemasks = ispec.restore_results(dump_file)

elif atmos_iter_method == "lbl":

    # In 3_atmos, the Sun dump file uses this name (without the _q2 suffix)
    sun_dump_file = output_atmos_params_dumpfile_path
    params_sun = {
        "teff": teff_q2,   # 5777
        "logg": logg_q2,
        "MH":   0.0,     # 0.0
        "alpha": 0.0,    # 0.0
        "vmic": vt_q2,   # estimate_vmic(5777, 4.44, 0)
    }
    errors_sun = {"teff": 0.0, "logg": 0.0, "MH": 0.0, "alpha": 0.0, "vmic": 0.0}
    status_sun = {"converged": True, "method": "fixed_solar_params_for_q2_reference"}

    # The dump structure must be consistent with the 8 return values
    # expected by restore_results in 4_ele
    empty = None
    dump_payload = (
        params_sun,
        errors_sun,
        status_sun,
        empty,          # x_over_h_fe (placeholder)
        empty,          # selected_x_over_h_fe (placeholder)
        None,           # fitted_lines_params_fe (placeholder)
        linemasks_fe,   # used_linemasks_fe (providing Fe masks is sufficient)
        None            # spec_abund_fe (placeholder)
    )

    ispec.save_results(sun_dump_file, dump_payload)
    print(f"[0_Sun][Part3] Saved Sun dump -> {sun_dump_file}")

## Part4: ele_abundance

In [ ]:
# =============== 1. Get paths & read data =================
# Get the spectrum path
star_spectrum_file = spectrum_norm_path
star_spectrum = ispec.read_spectrum(star_spectrum_file)

# =============== 2. Read linelist =================
linemask_output_folder = output_folder + "/linemasks"
# Linemasks with cross-matching and fitting information, used by model_spectrum_from_ew:
linemasks = ispec.read_line_regions(
    linemask_output_folder + f"/{target}_melendez2014_star_fitted_linemasks.txt"
)
print(f"Number of lines in the linemask: {len(linemasks)}")

# =============== 3. Load dump file containing atmospheric parameter fitting results =================
# params, errors, status, x_over_h, selected_x_over_h, fitted_lines_param, used_linemasks = ispec.restore_results(dump_file)
dump_file = output_atmos_params_dumpfile_path
if atmos_iter_method == "glb":
    params, errors, status_fe, x_over_h_fe, selected_x_over_h_fe, \
        fitted_lines_param_fe, used_linemasks_fe, spec_abund_fe = ispec.restore_results(dump_file)
elif atmos_iter_method == "lbl":
    payload = ispec.restore_results(dump_file)
    params = payload[0]
    errors = payload[1]
    status_q2 = payload[2]

# =============== 4. Retrieve atmospheric parameters =================
teff, tefferr = params['teff'], errors['teff']
logg, loggerr = params['logg'], errors['logg']
mh, mherr = params['MH'], errors['MH']
vmic, vmicerr = params['vmic'], errors['vmic']
alpha, alphaerr = params['alpha'], errors['alpha']
print("==================== Fitted stellar parameters =================")
print(f"Teff = {teff:.2f} +/- {tefferr:.2f} K")
print(f"logg = {logg:.2f} +/- {loggerr:.2f} dex")
print(f"[M/H] = {mh:.2f} +/- {mherr:.2f} dex")
print(f"vmic = {vmic:.2f} +/- {vmicerr:.2f} km/s")

# =============== Plotting (all elements) ===============
base_output_dir = output_folder + "/figs_ele_GaussianFits"
os.makedirs(base_output_dir, exist_ok=True)
deleted_elelines_folder_path = base_output_dir + "/deleted"
modified_elelines_folder_path = base_output_dir + "/modified"
os.makedirs(deleted_elelines_folder_path, exist_ok=True)
os.makedirs(modified_elelines_folder_path, exist_ok=True)

w_range = 0.25  # nm

# Define Gaussian function
def gaussian(x, mu, sig, A, baseline):
    return baseline + A * np.exp(-(x - mu)**2 / (2 * sig**2))

# Loop over each element
elements = np.unique(linemasks['element'])
for ele in elements:
    if "Fe" in ele:   # Skip iron to avoid duplicating Fe analysis
        continue

    ele_lines = linemasks[linemasks['element'] == ele]
    
    if len(ele_lines) == 0:
        continue
    
    # Create subfolder for the current element
    ele_output_dir = os.path.join(base_output_dir, ele.replace(" ", "_"))
    os.makedirs(ele_output_dir, exist_ok=True)

    # Loop over all spectral lines of this element
    for idx, line in enumerate(ele_lines):
        mu = line['mu']
        sig = line['sig']
        A = line['A']
        baseline = line['baseline']
        
        if sig == 0 or mu == 0:  # Invalid fit
            continue

        # Extract the spectral segment
        mask = (
            (star_spectrum['waveobs'] >= mu - w_range) &
            (star_spectrum['waveobs'] <= mu + w_range)
        )
        wave = star_spectrum['waveobs'][mask]
        flux = star_spectrum['flux'][mask]

        # Gaussian fit curve
        fit_x = np.linspace(mu - w_range, mu + w_range, 300)
        fit_y = gaussian(fit_x, mu, sig, A, baseline)

        # Plot
        plt.figure(figsize=(8, 5), dpi=128)
        plt.plot(wave, flux, label='Observed Spectrum', color='blue', lw=0.7)
        plt.plot(fit_x, fit_y, '--', label='Gaussian Fit', color='red')
        plt.axvline(mu, color='orange', linestyle=':', label=f"$\\mu$ = {mu:.3f} nm")
        plt.title(f"Gaussian Fit: {line['element']} {line['wave_A']:.2f} Å", fontsize=16)
        plt.xlabel("Wavelength (nm)", fontsize=16)
        plt.ylabel("Normalized Flux", fontsize=16)
        plt.xticks(fontsize=13)
        plt.yticks(fontsize=13)
        plt.ticklabel_format(style='plain', axis='x')

        # Annotations
        text = (
            f"log(gf) = {line['loggf']:.2f}\n"
            f"EP = {line['lower_state_eV']:.2f} eV\n"
            f"EW = {line['ew']:.1f} mÅ"
        )
        plt.text(
            0.75, 0.05, text,
            transform=plt.gca().transAxes,
            fontsize=13,
            bbox=dict(facecolor='white', alpha=0.8)
        )
        plt.legend(loc="lower left", fontsize=13)
        plt.tight_layout()

        # Save figure
        out_path = os.path.join(ele_output_dir, f"{ele}_{idx+1}_{line['wave_A']:.2f}.png")
        plt.savefig(out_path)
        plt.close()

    print(f"Element {ele}: plotted {len(ele_lines)} figures, saved in {ele_output_dir}/")

print(f"\nPlotting completed. All element results are saved in {base_output_dir}/")

## After completion, manually remove bad lines and then rewrite the linemasks

In [ ]:
# Filter invalid lines:
linemasks = linemasks[linemasks['wave_nm'] > 0]   # Successfully cross-matched
linemasks = linemasks[linemasks['ew'] > 0]        # Valid equivalent width (EW)

#--- Determining abundances by EW of the previously fitted lines ---------------
code = "moog"
# Parameters
teff = teff
logg = logg
MH = mh
alpha = alpha
microturbulence_vel = vmic  # km/s

# ========== Read model atmospheres and solar abundances ==========
# Selected model atmosphere and solar abundances
# Atmospheric model grids (different users adopt different grids; mainly differ in grid density/speed)
#model = ispec_dir + "/input/atmospheres/MARCS/"     # Very large grid
model = ispec_dir + "/input/atmospheres/MARCS.GES/" # Smaller grid with preliminary interpolation
#model = ispec_dir + "/input/atmospheres/MARCS.APOGEE/"
#model = ispec_dir + "/input/atmospheres/ATLAS9.APOGEE/"
#model = ispec_dir + "/input/atmospheres/ATLAS9.Castelli/"
#model = ispec_dir + "/input/atmospheres/ATLAS9.Kurucz/"
#model = ispec_dir + "/input/atmospheres/ATLAS9.Kirby/"

# Solar abundance reference (different compilations)
if "ATLAS" in model:
    solar_abundances_file = ispec_dir + "/input/abundances/Grevesse.1998/stdatom.dat"
else:
    # MARCS
    solar_abundances_file = ispec_dir + "/input/abundances/Grevesse.2007/stdatom.dat"
#solar_abundances_file = ispec_dir + "/input/abundances/Asplund.2005/stdatom.dat"
#solar_abundances_file = ispec_dir + "/input/abundances/Asplund.2009/stdatom.dat"
#solar_abundances_file = ispec_dir + "/input/abundances/Anders.1989/stdatom.dat"

# ========== Load models ==========
# Load model atmospheres
modeled_layers_pack = ispec.load_modeled_layers_pack(model)
# Load SPECTRUM solar abundances
solar_abundances = ispec.read_solar_abundances(solar_abundances_file)

# Validate parameters
# Check whether the parameters fall within the model grid
if not ispec.valid_atmosphere_target(
    modeled_layers_pack,
    {'teff': teff, 'logg': logg, 'MH': MH, 'alpha': alpha}
):
    msg = (
        "The specified effective temperature, gravity (log g) and metallicity [M/H] "
        "fall out of the atmospheric models."
    )
    print(msg)

# Prepare atmosphere model
# Interpolate the model atmosphere
atmosphere_layers = ispec.interpolate_atmosphere_layers(
    modeled_layers_pack,
    {'teff': teff, 'logg': logg, 'MH': MH, 'alpha': alpha},
    code=code
)

In [ ]:
if atmos_iter_method == "lbl":
    # ===================== Part4: Generate / update solar_lines_instru & solar_abund_instru =====================
    # --- 0) Paths: always write to iSpec/input so that 4_ele_abundance can read directly ---
    solar_lines_csv = os.path.join(ispec_dir, "input", "solar_lines_instru.csv")
    solar_abund_csv = os.path.join(ispec_dir, "input", "solar_abund_instru.csv")

    # --- 1) Read back the q2 atmospheric parameters from Part3 ---

    # --- 2) iSpec requirements: solar abundance file + atmosphere layers ---
    solar_abundances = ispec.read_solar_abundances(solar_abundances_file)
    modeled_layers_pack = ispec.load_modeled_layers_pack(model)
    atmosphere_layers = ispec.interpolate_atmosphere_layers(
        modeled_layers_pack,
        {"teff": teff, "logg": logg, "MH": mh, "alpha": alpha}
    )

    # --- 3) Use the already measured linemasks (with ew/wave/loggf/EP, etc.) to compute abundances element by element ---
    # Here we assume that linemasks already exist (the structured array obtained from fit_lines + crossmatch)
    def norm_elem_ispec(e):
        s = str(e).strip()
        # In 4_ele_abundance the format is "Fe 1"/"Fe 2"; keep it consistent here
        return s

    elem_norm = np.array([norm_elem_ispec(e) for e in linemasks["element"]])
    elements = sorted(set(elem_norm.tolist()))

    solar_lines = []
    for ele in elements:
        sel = (elem_norm == ele)
        lm_ele = linemasks[sel]
        if len(lm_ele) == 0:
            continue

        spec_abund, normal_abund, x_over_h, x_over_fe = ispec.determine_abundances(
            atmosphere_layers,
            teff, logg, mh, alpha,
            lm_ele,
            solar_abundances,
            microturbulence_vel=microturbulence_vel,
            verbose=0,
            code=code
        )

        spec_abund = np.asarray(spec_abund, dtype=float)  # logeps - 12
        x_over_h   = np.asarray(x_over_h, dtype=float)    # [X/H] relative to solar_abundances_file

        for lm, logeps_m12, xh in zip(lm_ele, spec_abund, x_over_h):
            if not (np.isfinite(logeps_m12) and np.isfinite(xh)):
                continue
            solar_lines.append({
                "instrument": instrument,          # Must match the filtering keys used in 4_
                "resolution": int(from_resolution),
                "solar_label": str(solar_label),   # Only for your own version tracking (not used in 4_)
                "element": ele,
                "wave_A": float(lm["wave_A"]),
                "EP": float(lm["lower_state_eV"]),
                "loggf": float(lm["loggf"]),
                "ew_mA": float(lm["ew"]),
                "logeps_sun": float(logeps_m12 + 12.0),
                "[X/H]_sun_Grevesse": float(xh),
            })

    df_solar_lines = pd.DataFrame(solar_lines)
    print("Solar per-line rows:", len(df_solar_lines))

    # --- 4) Update master solar_lines_instru: keep only one solar library per instrument + resolution ---
    if os.path.exists(solar_lines_csv):
        master = pd.read_csv(solar_lines_csv)
        master = master[~((master["instrument"] == instrument) & (master["resolution"] == int(from_resolution)))]
        master = pd.concat([master, df_solar_lines], ignore_index=True)
    else:
        master = df_solar_lines

    master.to_csv(solar_lines_csv, index=False)
    print("Saved solar_lines_instru ->", solar_lines_csv, "rows:", len(master))

    # --- 5) Generate element-level solar_abund_instru (the desired solar_abund_instru) ---
    g = df_solar_lines.groupby("element")
    df_solar_abund = pd.DataFrame({
        "instrument": instrument,
        "resolution": int(from_resolution),
        "solar_label": str(solar_label),
        "element": g.size().index,
        "n_lines": g.size().values,
        "logeps_sun_mean": g["logeps_sun"].mean().values,
        "[X/H]_sun_mean_Grevesse": g["[X/H]_sun_Grevesse"].mean().values,
        "std_[X/H]_sun": g["[X/H]_sun_Grevesse"].std(ddof=1).values,
    })

    # Similarly overwrite/update by instrument + resolution
    if os.path.exists(solar_abund_csv):
        masterA = pd.read_csv(solar_abund_csv)
        masterA = masterA[~((masterA["instrument"] == instrument) & (masterA["resolution"] == int(from_resolution)))]
        masterA = pd.concat([masterA, df_solar_abund], ignore_index=True)
    else:
        masterA = df_solar_abund

    masterA.to_csv(solar_abund_csv, index=False)
    print("Saved solar_abund_instru ->", solar_abund_csv, "rows:", len(masterA))

In [ ]:
if atmos_iter_method == "glb":
    # ========= Step 1: Split Fe I / Fe II from the step-1 results =========
    # x_over_h_fe: per-line [Fe/H] from step 1 (may contain NaN)
    # selected_x_over_h_fe: [mask_fe1, mask_fe2], two boolean arrays indicating
    #                       whether each line belongs to Fe I / Fe II
    # status_fe['fe1_lines'], status_fe['fe2_lines'] are for information only

    xh_all = np.asarray(x_over_h_fe, dtype=float)
    spec_all = np.asarray(spec_abund_fe, dtype=float)

    mask_valid_xh = np.isfinite(xh_all)

    mask_fe1 = np.asarray(selected_x_over_h_fe[0], dtype=bool) & mask_valid_xh
    mask_fe2 = np.asarray(selected_x_over_h_fe[1], dtype=bool) & mask_valid_xh

    FeI_vals  = xh_all[mask_fe1]
    FeII_vals = xh_all[mask_fe2]
    FeI_spec  = spec_all[mask_fe1]
    FeII_spec = spec_all[mask_fe2]

    FeI_mean  = float(np.nanmean(FeI_vals))  if FeI_vals.size  else np.nan
    FeI_std   = float(np.nanstd(FeI_vals))   if FeI_vals.size  else np.nan
    FeI_mean_abund = float(np.nanmean(FeI_spec))  if FeI_spec.size else np.nan
    FeI_std_abund  = float(np.nanstd(FeI_spec))   if FeI_spec.size else np.nan
    FeI_log_eps = (FeI_mean_abund + 12.0) if FeI_spec.size else np.nan

    FeII_mean = float(np.nanmean(FeII_vals)) if FeII_vals.size else np.nan
    FeII_std  = float(np.nanstd(FeII_vals))  if FeII_vals.size else np.nan
    FeII_mean_abund = float(np.nanmean(FeII_spec)) if FeII_spec.size else np.nan
    FeII_std_abund  = float(np.nanstd(FeII_spec))  if FeII_spec.size else np.nan
    FeII_log_eps = (FeII_mean_abund + 12.0) if FeII_spec.size else np.nan

    # Reference iron for [X/Fe] (neutral → Fe I; ionized → Fe II)
    def pick_fe_ref_mean(elem: str) -> float:
        """elem like 'Ti 1'/'Ti 2'; use Fe II for ' 2', otherwise Fe I."""
        s = str(elem).strip()
        return FeII_mean if s.endswith(" 2") else FeI_mean

    # ========= Step 2: Compute only non-Fe elements; fix metallicity MH to step-1 value =========
    elements_all = np.unique(linemasks['element'])
    elements = [e for e in elements_all if not str(e).startswith("Fe")]

    results = []

    for ele in elements:
        submask = linemasks[linemasks['element'] == ele]
        if len(submask) == 0:
            continue

        spec_abund, normal_abund, x_over_h, _ = ispec.determine_abundances(
            atmosphere_layers,
            teff, logg, MH, alpha,            # <<< Fix metallicity to step-1 [Fe/H]
            submask,
            solar_abundances,
            microturbulence_vel=microturbulence_vel,
            verbose=0,
            code=code
        )

        xh = x_over_h[np.isfinite(x_over_h)]
        xh_mean = float(np.nanmean(xh)) if xh.size else np.nan
        xh_std  = float(np.nanstd(xh))  if xh.size else np.nan

        # Compute [X/Fe] using Fe reference (by ionization stage)
        fe_ref = pick_fe_ref_mean(ele)
        xfe_mean = (xh_mean - fe_ref) if (np.isfinite(xh_mean) and np.isfinite(fe_ref)) else np.nan
        xfe_std  = xh_std  # For error propagation, could use sqrt(xh_std**2 + fe_ref_std**2)

        results.append({
            "element": str(ele),
            "n_lines": int(len(submask)),
            "[X/H]": xh_mean,
            "std_[X/H]": xh_std,
            "[X/Fe]": xfe_mean, 
            "mean_abund": float(np.nanmean(spec_abund)) if len(spec_abund) else np.nan,
            "std_abund":  float(np.nanstd(spec_abund))  if len(spec_abund) else np.nan,
            "log_eps": (float(np.nanmean(spec_abund)) + 12.0) if len(spec_abund) else np.nan,
            "normal_abund": normal_abund,
            "x_over_h": x_over_h,
            "x_over_fe": x_over_h - fe_ref if np.isfinite(fe_ref) else np.nan,
        })

    # Also add Fe I / Fe II from step 1 into the results table (no recomputation)
    results.append({
        "element": "Fe I",
        "n_lines": int(mask_fe1.sum()),
        "[X/H]": FeI_mean,
        "std_[X/H]": FeI_std,
        "[X/Fe]": 0.0,
        "mean_abund": FeI_mean_abund,
        "std_abund":  FeI_std_abund,
        "log_eps": FeI_log_eps,
        "normal_abund": np.nan,
        "x_over_h": FeI_vals,
        "x_over_fe": 0.0,
    })
    results.append({
        "element": "Fe II",
        "n_lines": int(mask_fe2.sum()),
        "[X/H]": FeII_mean,
        "std_[X/H]": FeII_std,
        "[X/Fe]": 0.0,
        "mean_abund": FeII_mean_abund,
        "std_abund":  FeII_std_abund,
        "log_eps": FeII_log_eps,
        "normal_abund": np.nan,
        "x_over_h": FeII_vals,
        "x_over_fe": 0.0,
    })

    df_results = pd.DataFrame(results)

    # Sort: put Fe I and Fe II at the top
    idx_fei  = df_results.index[df_results["element"] == "Fe I"].tolist()
    idx_feii = df_results.index[df_results["element"] == "Fe II"].tolist()
    idx_others = df_results.index[(df_results["element"] != "Fe I") & (df_results["element"] != "Fe II")].tolist()
    df_results = df_results.loc[idx_fei + idx_feii + idx_others].reset_index(drop=True)

    output_abundances_result_path = output_folder + "abundances_" + target + ".csv"
    print(df_results)
    df_results.to_csv(output_abundances_result_path, index=False)
    print(f"\nAbundance calculation completed (synthesis with fixed metallicity MH={MH:.3f} dex; "
          f"[X/Fe] uses Fe I for neutral lines and Fe II for ionized lines). "
          f"Results saved to {output_abundances_result_path}")

In [ ]:
if atmos_iter_method == "glb":
    # ================== Generate instrument-wise solar abundance summary table ==================

    # 1) Prepare a dictionary for this row, including instrument / solar_label
    row_dict = {
        "instrument": instrument,
        "solar_label": solar_label,
        'resolution': from_resolution,
    }

    # 2) Expand [X/H] and std_[X/H] for each element in df_results into columns
    #    Example column names: Fe_I_XH, Fe_I_eXH, Mg_1_XH, Mg_1_eXH
    metrics = {
        "[X/H]": "XH",
        "std_[X/H]": "eXH",
    }

    for _, r in df_results.iterrows():
        elem_raw = str(r["element"]).strip()      # e.g. "Fe I"
        # Replace spaces with underscores to avoid awkward column names
        elem_col = elem_raw.replace(" ", "_")     # "Fe_I"

        for col_name, short_name in metrics.items():
            if col_name not in r:
                continue
            new_col = f"{elem_col}_{short_name}"
            row_dict[new_col] = r[col_name]

    # 3) Convert this row into a DataFrame
    df_this_sun = pd.DataFrame([row_dict])

    # 4) Read / write the master table solar_abund_instru.csv
    solar_abund_csv = os.path.join(ispec_dir, "input", "solar_abund_instru.csv")
    os.makedirs(os.path.dirname(solar_abund_csv), exist_ok=True)

    if os.path.exists(solar_abund_csv):
        master = pd.read_csv(solar_abund_csv)
        print(f"Loaded existing solar_abund_instru.csv with {len(master)} rows")

        # Remove old rows with the same instrument + solar_label (useful when re-running the same solar spectrum)
        mask = (master["instrument"] == instrument) & (master["solar_label"] == solar_label)
        if mask.any():
            print(f"Removing old results: instrument={instrument}, solar_label={solar_label}, rows={mask.sum()}")
            master = master[~mask]

        master = pd.concat([master, df_this_sun], ignore_index=True)
    else:
        print("solar_abund_instru.csv not found, creating a new one.")
        master = df_this_sun.copy()

    master.to_csv(solar_abund_csv, index=False)
    print(f"Updated {solar_abund_csv}, total number of solar spectra records: {len(master)}")

## Generate the instrument-wise solar per-line summary table: solar_lines_instru.csv
- This table records the abundance results for the Sun treated as a star, on a line-by-line basis.
- Each row corresponds to one spectral line, grouped by instrument + resolution, and stores element, wavelength, EW, and [X/H]. 
- This enables strict line-by-line differential analysis by subtracting the solar reference obtained with the same instrument when analyzing any star.

In [ ]:
if atmos_iter_method == "glb":
    # ================== 0. Initialize the per-line solar library ==================
    solar_lines = []

    # ================== 1. Fe I / Fe II per-line ==================
    # Re-compute per-line Fe abundances using the fitted atmospheric parameters
    # and the final used_linemasks_fe
    fe_spec_abund, fe_normal_abund, fe_x_over_h, _ = ispec.determine_abundances(
        atmosphere_layers,
        teff, logg, MH, alpha,
        used_linemasks_fe,          # only the final retained Fe lines (e.g. 43 lines)
        solar_abundances,
        microturbulence_vel=microturbulence_vel,
        verbose=0,
        code=code
    )

    xh_all_fe   = np.asarray(fe_x_over_h,   dtype=float)
    spec_all_fe = np.asarray(fe_spec_abund, dtype=float)

    print(
        "Fe lines: used_linemasks_fe =", len(used_linemasks_fe),
        "spec_all_fe =", len(spec_all_fe),
        "xh_all_fe =", len(xh_all_fe)
    )

    for lm, logeps, xh in zip(used_linemasks_fe, spec_all_fe, xh_all_fe):
        # Ensure only valid lines are stored
        if not (np.isfinite(logeps) and np.isfinite(xh)):
            continue

        solar_lines.append({
            "instrument": instrument,
            "solar_label": solar_label,
            "resolution": from_resolution,
            "element": lm["element"],          # "Fe I" or "Fe II"
            "wave_A": lm["wave_A"],
            "EP": lm["lower_state_eV"],
            "loggf": lm["loggf"],
            "ew_mA": lm["ew"],
            "logeps_sun": logeps + 12.0,       # spec_abund is logeps - 12
            "[X/H]_sun_Grevesse": xh,
        })

    print(
        "Fe lines (used_linemasks_fe):", len(used_linemasks_fe),
        "Fe lines stored:", sum(row["element"].startswith("Fe") for row in solar_lines)
    )

    # ================== 2. Non-Fe elements, line by line ==================
    elements_all = np.unique(linemasks['element'])
    elements = [e for e in elements_all if not str(e).startswith("Fe")]

    results = []

    for ele in elements:
        submask = linemasks[linemasks['element'] == ele]
        if len(submask) == 0:
            continue

        spec_abund, normal_abund, x_over_h, _ = ispec.determine_abundances(
            atmosphere_layers,
            teff, logg, MH, alpha,
            submask,
            solar_abundances,
            microturbulence_vel=microturbulence_vel,
            verbose=0,
            code=code
        )

        # ====== (A) Original element-wise mean [X/H]: keep unchanged ======
        xh = x_over_h[np.isfinite(x_over_h)]
        xh_mean = float(np.nanmean(xh)) if xh.size else np.nan
        xh_std  = float(np.nanstd(xh))  if xh.size else np.nan
        # ... original results.append({...}) ...

        # ====== (B) Write per-line entries into solar_lines (line-by-line solar library) ======
        xh_all   = np.asarray(x_over_h,   dtype=float)
        spec_all = np.asarray(spec_abund, dtype=float)

        for lm, logeps, xh_line in zip(submask, spec_all, xh_all):
            if not (np.isfinite(logeps) and np.isfinite(xh_line)):
                continue
            solar_lines.append({
                "instrument": instrument,
                "solar_label": solar_label,
                "resolution": from_resolution,
                "element": lm["element"],      # e.g. "Mg 1"
                "wave_A": lm["wave_A"],
                "EP": lm["lower_state_eV"],
                "loggf": lm["loggf"],
                "ew_mA": lm["ew"],
                "logeps_sun": logeps + 12.0,
                "[X/H]_sun_Grevesse": xh_line,
            })

    # ================== 3. Save the per-line solar library ==================
    df_solar_lines = pd.DataFrame(solar_lines)

    solar_lines_csv = os.path.join(ispec_dir, "input", "solar_lines_instru.csv")
    if os.path.exists(solar_lines_csv):
        master_lines = pd.read_csv(solar_lines_csv)
        mask = (
            (master_lines["instrument"] == instrument) &
            (master_lines["solar_label"] == solar_label) &
            (master_lines["resolution"] == from_resolution)
        )
        if mask.any():
            print(f"Removing old solar per-line records: {mask.sum()} rows")
            master_lines = master_lines[~mask]
        master_lines = pd.concat([master_lines, df_solar_lines], ignore_index=True)
    else:
        master_lines = df_solar_lines

    master_lines.to_csv(solar_lines_csv, index=False)
    print(
        "Updated per-line solar library:", solar_lines_csv,
        "total number of rows:", len(master_lines)
    )